# chl59-8b_1p Analysis

- Creator: Sebastian Birk (sb75@sanger.ac.uk)
- Date of Creation: 21.02.2025
- Date of Last Modification: 24.06.2026 (by Sebastian Birk; sb75@sanger.ac.uk)

Run this notebook in the terra-rapids UV environment, installable from ```/nfs/team361/sb75/terra-reproducibility/envs/terra-rapids.sh```:

```source /nfs/team361/sb75/.venvs/terra-rapids/bin/activate```

# 1. Setup

## 1.1 Load Libraries 

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
sys.path.append("../../utils")

In [ ]:
import logging
import math
import os
import pickle
import random
import re
import warnings
from collections import defaultdict
from datetime import datetime
from itertools import chain, compress
from pathlib import Path

import anndata as ad
import celltypist
import dask
import matplotlib as mpl
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rapids_singlecell as rsc
import scanpy as sc
import scib_metrics
import seaborn as sns
import torch

dask.config.set({"dataframe.query-planning": True})
import dask.dataframe.dask_expr
import squidpy as sq
import rapids_singlecell as rsc

from datasets import concatenate_datasets, load_from_disk
from scib_metrics import ilisi_knn
from scib_metrics.nearest_neighbors import pynndescent
from scipy.sparse import csr_matrix
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from app.inference import get_gene_embed, get_average_gene_embed, infer, embed_dataset
from app.training import train
from terra.datasets import prepare_dataset
from terra.utils.config import create_params_from_YAML_wandb_config
from terra.utils.distributed import init_distributed
from terra.utils.embedding import collect_adata_from_folder
from terra.utils.evaluation import clustering_metrics, classification_metrics

from analysis_utils import *

## 1.2 Load Config/Params

In [ ]:
# Set params
dataset = 'chl59-8b_1p'
tokenizer = 'cell_graph'
load_timestamp = '24062026_152822_800'
read_checkpoint = 'terra-latest.pth.tar' # NMI 0.705; ARI 0.385
emb_layer = None
top_k = None
exclude_unshared_tokens = False # True, False
agg_type = 'avg' # 'avg', 'weighted_avg', 'cls'
batch_size = 128

In [ ]:
# Define paths
artifacts_folder_path = '/nfs/team361/sb75/terra-reproducibility/artifacts'
datasets_folder_path = '/nfs/team361/sb75/DATASETS'
raw_data_folder_path = f'/nfs/team361/sb75/DATASETS/silver/{dataset}'

# Define folder path for saving and loading artifacts
model_folder_path = f'{artifacts_folder_path}/{dataset}/{load_timestamp}'

In [ ]:
# Set up logger
rank = 0
logging.basicConfig()
logger = logging.getLogger()
logger.setLevel(logging.INFO if rank == 0 else logging.ERROR)

In [ ]:
# Load config
params = create_params_from_YAML_wandb_config(f'{artifacts_folder_path}/{dataset}/{load_timestamp}/params.yaml', logger)

if read_checkpoint:
    params['state']['read_checkpoint'] = read_checkpoint
    params['state']['load_checkpoint'] = True

layer = params['meta']['enc_depth']
params['data']['batch_size'] = batch_size

old_tokenized_data_folder_path = Path(params["data"]["tokenized_data_folder_path"])
filename = old_tokenized_data_folder_path.name
_, after_none = filename.split("None", 1)
tokenized_data_folder_path = old_tokenized_data_folder_path.parent.parent / "hst_corpus_110m" / f"hst_corpus_110m_32_None{after_none}"

print(params)

## 1.3 Run Notebook Setup

In [ ]:
os.makedirs(f'{model_folder_path}/figures', exist_ok=True)

In [ ]:
warnings.filterwarnings("ignore", category=ResourceWarning)

In [ ]:
plt.rcParams['font.size'] = 5
plt.rcParams['text.usetex'] = False
plt.rcParams['svg.fonttype'] = 'none'

In [ ]:
# Improve reproducibility of UMAP and Leiden
os.environ['NUMBA_CPU_NAME'] = 'generic'

In [ ]:
sc.settings.figdir = f'{model_folder_path}/figures'

sc.settings.set_figure_params(
    dpi=300,
    dpi_save=300,
    figsize=(6, 4),
    facecolor='white',
    fontsize=10
)

## 1.4 Define Functions

In [ ]:
def make_large_cmap(n):
    base = mpl.colormaps["turbo"].resampled(n)   # replaces plt.cm.get_cmap("turbo", n)
    return [mcolors.to_hex(base(i)) for i in range(n)]

In [ ]:
def run_inference(dataset_name,
                  params,
                  read_checkpoint,
                  obs_cols,
                  emb_layers,
                  #n_included_cells_list,
                  top_k,
                  rerun_inference=False,
                  write_to_disk=True):
    # Load inference results if they exist
    if os.path.exists(f'{model_folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad') and not rerun_inference:
        adata = sc.read_h5ad(f'{model_folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')
    else:
        # Load data
        if params['data']['precomputed_split']:
            test_dataset, _, _ = prepare_dataset(params)
        else:
            _, _, test_dataset = prepare_dataset(params)
        test_dataset
        
        # Run model inference
        adata = infer(params,
                      test_dataset,
                      load_folder_path=model_folder_path,
                      dataset_ids=list(set([batch_id.split('_')[0] for batch_id in params['data']['test_batch_ids']])),
                      obs_cols=obs_cols,
                      emb_layers=emb_layers,
                      agg_type='avg',
                      #n_included_cells_list=n_included_cells_list,
                      top_k=top_k,
                      masked_tokens=None,
                      agg_excluded_genes=None,
                      include_spatial_cell_emb=True,
                      cell_gene_ids=[],
                      neighborhood_gene_ids=[],
                      ignore_spc_tokens=True)

    # Extract batch from cell ID
    adata.obs["batch"] = [cell_id.split('_')[1].replace('batch', '') for cell_id in adata.obs.index.tolist()]
    
    # Remove missing ground truth
    #for obs_col in obs_cols:
    #    adata = adata[~adata.obs[obs_col].isna()]

    if write_to_disk:
        # Save adata
        adata.write(f'{model_folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')

    return adata

In [ ]:
def compute_embedding_similarity(adata,
                                 emb_layer,
                                 emb_keys=['cell', 'neighborhood'],
                                 label='niche_type',
                                 write_to_disk=True,
                                 ):
    for emb_key in emb_keys:
        similarity_stats = compute_similarity_stats(
            adata,
            n_samples=None,
            type_key=label,
            embedding_key=f'{emb_key}_emb_layer_{emb_layer}')

        avg_cos_sim = np.mean(similarity_stats["Mean Similarity"])
    
        cosine_df = pd.DataFrame({
            'metric': [f'avg_{label}_cos_sim'],
            'value': [avg_cos_sim]
        })
        print(f'Average {label} cosine similarity is {avg_cos_sim}.')
        cosine_df.to_csv(f'{model_folder_path}/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_avg_{label}_cos_sim.csv',
                         index=False)
        
        plot_similarity_heatmaps(
            similarity_stats,
            save_file_path=f'{model_folder_path}/figures/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{emb_key}_emb_{label}_cos_sim.svg')

In [ ]:
def compute_umap(adata,
                 emb_keys,
                 emb_layer,
                 write_to_disk=True):
    for emb_key in emb_keys:
        if f'{emb_key}_connectivities' not in adata.obsp.keys():
            # Compute UMAP representation for aggregated gene embeddings
            rsc.pp.neighbors(adata,
                             n_neighbors=15,
                             use_rep=f'{emb_key}_emb_layer_{emb_layer}',
                             key_added=emb_key)
        if f'{emb_key}_umap' not in adata.obsm.keys():
            rsc.tl.umap(adata,
                        neighbors_key=emb_key,
                        key_added=f'{emb_key}_umap')
        
    if write_to_disk:
        # Save adata
        adata.write(f'{model_folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')
                
    return adata

In [ ]:
def compute_latent_clusters(adata,
                            emb_keys=['cell', 'neighborhood'],
                            obs_col='niche_type',
                            n_unique_vals=None,
                            latent_leiden_resolution_cell=1.0,
                            latent_leiden_resolution_neighborhood=0.4,
                            latent_leiden_resolution_agg=0.4,
                            write_to_disk=True):
    for emb_key in emb_keys:
        latent_leiden_resolutions = []
        if emb_key == 'cell' or 'ctx_1' in emb_key:
            latent_leiden_resolution = latent_leiden_resolution_cell
        elif emb_key == 'neighborhood' or 'ctx' in emb_key:
            latent_leiden_resolution = latent_leiden_resolution_neighborhood
        elif emb_key == 'agg':
            latent_leiden_resolution = latent_leiden_resolution_agg        
        adata.obsm['X_umap'] = adata.obsm[f'{emb_key}_umap']
        # Perform clustering until number of ground truth niches is matched
        i = 0
        if not n_unique_vals:
            n_unique_vals = len(adata.obs[obs_col].value_counts() < 100)
        print(f'Target number of unique values: {n_unique_vals}.')
        n_latent_clusters = 0
        while (n_latent_clusters != n_unique_vals):
            if latent_leiden_resolution in latent_leiden_resolutions:
                latent_leiden_resolution += random.uniform(0.0001, 0.001)
            latent_leiden_resolutions.append(latent_leiden_resolution)
            print(f'Leiden resolution: {latent_leiden_resolution}.')
            latent_cluster_key = f'{emb_key}_emb_leiden_res{str(latent_leiden_resolution).replace(".", "_")}'
            i += 1
            rsc.tl.leiden(adata,
                          neighbors_key=emb_key,
                          key_added=latent_cluster_key,
                          resolution=latent_leiden_resolution)
        
            n_latent_clusters = len(adata.obs[latent_cluster_key].value_counts() < 100)
            print(f"Number of latent clusters {n_latent_clusters}.")
            if n_latent_clusters == n_unique_vals:
                break
            elif n_latent_clusters < (n_unique_vals - 8):
                latent_leiden_resolution += 0.1
            elif n_latent_clusters < (n_unique_vals - 5):
                latent_leiden_resolution += 0.05
            elif n_latent_clusters < (n_unique_vals - 3):
                latent_leiden_resolution += 0.02
            elif n_latent_clusters < n_unique_vals:
                latent_leiden_resolution += 0.001
            elif n_latent_clusters > (n_unique_vals + 8):
                latent_leiden_resolution -= 0.1
            elif n_latent_clusters > (n_unique_vals + 5):
                latent_leiden_resolution -= 0.05
            elif n_latent_clusters > (n_unique_vals + 3):
                latent_leiden_resolution -= 0.02
            else:
                latent_leiden_resolution -= 0.001
        
        # Compute metrics and store to disk
        labels = adata.obs[obs_col]
        labels_pred = adata.obs[latent_cluster_key]
        nmi = normalized_mutual_info_score(labels, labels_pred, average_method="arithmetic")
        ari = adjusted_rand_score(labels, labels_pred)
        print(f'NMI is {nmi}. ARI is {ari}.')
        metrics_df = pd.DataFrame({
            'metric': ['NMI', 'ARI'],
            'value': [nmi, ari]
        })
        metrics_df.to_csv(f'{model_folder_path}/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_metrics.csv',
                          index=False)
        
        # Plot UMAPs
        os.makedirs(f'{model_folder_path}/figures', exist_ok=True)
        labels = ['batch', obs_col, latent_cluster_key]
        # Plot (aggregated) neighborhood gene embeddings, colored by labels
        for label in labels:
            if len(np.unique(adata.obs[label])) > 1:
                n_cats = adata.obs[label].nunique()
                palette = make_large_cmap(n_cats)
                sc.pl.umap(adata,
                           neighbors_key=emb_key,
                           title=(emb_key + "_emb_" + label if emb_key not in label else label),
                           color=label,
                           palette=(
                                custom_palette
                                if label == latent_cluster_key
                                else palette if label == obs_col
                                else None
                            ),
                           show=False)
                plt.savefig(f'{model_folder_path}/figures/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}{"_" + emb_key + "_emb" if emb_key not in label else ""}_{label}_umap.svg',
                            dpi=300,
                            bbox_inches='tight')
        
        # Plot niches in tissue
        batch_ids = adata.obs['batch'].unique().tolist()
        for i, batch_id in enumerate(batch_ids):
            adata_batch = adata[adata.obs['batch'] == batch_id]
            sc.pl.spatial(adata_batch,
                          color=latent_cluster_key,
                          palette=custom_palette,
                          spot_size=spot_sizes[i],
                          save=f'{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_batch_{batch_id}.svg')
            os.rename(f'{model_folder_path}/figures/show{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_batch_{batch_id}.svg',
                      f'{model_folder_path}/figures/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{latent_cluster_key}_batch_{batch_id}.svg'.replace("show", ""))
    
        if write_to_disk:
            # Save adata
            adata.write(f'{model_folder_path}/adata_{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}.h5ad')
    
    return adata, latent_cluster_key

In [ ]:
def mmd_comparable(adata, emb_key, batch_key, n_sub=2000, n_sigma=1000):
    # Normalize to zero mean, unit variance
    X = StandardScaler().fit_transform(adata.obsm[emb_key])
    
    # Compute sigma on normalized data
    idx = np.random.choice(len(X), min(n_sigma, len(X)), replace=False)
    sub = X[idx]
    dists = np.linalg.norm(sub[:, None] - sub[None, :], axis=2)
    sigma = np.median(dists[dists > 0])
    
    # Split by batch
    batches = adata.obs[batch_key].unique()
    mask0 = adata.obs[batch_key] == batches[0]
    mask1 = adata.obs[batch_key] == batches[1]
    b0, b1 = X[mask0], X[mask1]
    
    idx0 = np.random.choice(len(b0), min(n_sub, len(b0)), replace=False)
    idx1 = np.random.choice(len(b1), min(n_sub, len(b1)), replace=False)
    
    def rbf(A, B):
        d = np.linalg.norm(A[:, None] - B[None, :], axis=2)
        return np.exp(-d**2 / (2 * sigma**2))
    
    b0s, b1s = b0[idx0], b1[idx1]
    return rbf(b0s, b0s).mean() + rbf(b1s, b1s).mean() - 2 * rbf(b0s, b1s).mean()

In [ ]:
def compute_batch_integration_metrics(
    adata,
    emb_key,
    batch_key,
    dataset_name,
    model_folder_path,
    n_neighbors=90,
    sample_size=10000,
):
    """
    Compute batch integration metrics and save them to CSV.

    Metrics:
    - iLISI
    - ASW batch
    - MMD
    """
    results = []

    # iLISI
    batch = adata.obs[batch_key].values
    knn = pynndescent(adata.obsm[emb_key], n_neighbors=n_neighbors)
    ilisi = float(np.mean(ilisi_knn(knn, batch)))
    print(f"iLISI: {ilisi:.4f}")
    results.append({"metric": "iLISI", "score": ilisi})

    # ASW
    asw = silhouette_score(
        adata.obsm[emb_key],
        adata.obs[batch_key],
        metric="euclidean",
        sample_size=sample_size,
    )
    print(f"ASW batch: {asw:.4f}")
    results.append({"metric": "ASW", "score": asw})

    # MMD
    mmd_score = mmd_comparable(adata, emb_key, batch_key)
    print(f"MMD: {mmd_score:.4f}")
    results.append({"metric": "MMD", "score": mmd_score})

    # Save
    df = pd.DataFrame(results)
    output_path = f"{model_folder_path}/{read_checkpoint.split("-")[1].split(".")[0]}_{dataset_name}_{emb_key}_batch_integration_metrics.csv"
    df.to_csv(output_path, index=False)

    return df

# 2. Analysis

### 2.1 xhs49-1b_1p (Xenium Human Skin Melanoma)

#### 2.1.1 Define Dataset Config

In [ ]:
# Set dataset params
dataset_name = 'xhs49-1b_1p'
obs_cols = ['niche_name']
niche_key = 'niche_name'
latent_leiden_resolution_cell = 1.0
latent_leiden_resolution_neighborhood = 0.4
spot_sizes = [20]

In [ ]:
params["data"]["dataset_name"] = dataset_name
params["data"]["raw_data_folder_path"] = (
    params["data"]["raw_data_folder_path"].rsplit("/", 1)[0]
    + "/" + dataset_name)
params["data"]["tokenized_data_folder_path"] = str(tokenized_data_folder_path).rsplit("_", 1)[0] + '.dataset'
params["data"]["precomputed_split"] = f'/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/{params["data"]["tokenized_data_folder_path"].split("/")[-1].replace(".dataset", f"_precomputed_split_{dataset_name}.pkl")}'
params['data']['precomputed_n_nonzero_tokens'] = None
params['data']['test_batch_ids'] = [
    '49_batch0']
params['data']['val_batch_ids'] = None
if 'precomputed_epoch_n_nonzero_tokens' in params['data'].keys():
    del(params['data']['precomputed_epoch_n_nonzero_tokens'])
if 'precomputed_epoch_splits' in params['data'].keys():
    del(params['data']['precomputed_epoch_splits'])

#### 2.1.2 Run Model Inference

In [ ]:
adata = run_inference(dataset_name,
                      params,
                      read_checkpoint,
                      obs_cols=obs_cols,
                      emb_layers=[params['meta']['enc_depth']], # list(range(1, params['meta']['enc_depth'] + 1)),
                      top_k=top_k,
                      write_to_disk=True)

In [ ]:
# Filter cells without niche labels
adata = adata[~adata.obs[niche_key].isna()]

#### 2.1.3 Identify Niches

In [ ]:
adata = compute_umap(adata,
                     emb_keys=['neighborhood'], # ['agg'], ['neighborhood']
                     emb_layer=params['meta']['enc_depth'], # 'agg',
                     write_to_disk=True)

In [ ]:
adata, latent_cluster_key = compute_latent_clusters(
    adata,
    emb_keys=['neighborhood'], # ['agg'], ['neighborhood']
    obs_col=niche_key,
    n_unique_vals=12,
    latent_leiden_resolution_cell=latent_leiden_resolution_cell,
    latent_leiden_resolution_neighborhood=latent_leiden_resolution_neighborhood,
    write_to_disk=True)

#### 2.1.4 Plot Embedding Similarity Stats

In [ ]:
compute_embedding_similarity(adata,
                             emb_keys=['neighborhood'], # ['neighborhood'], ['agg']
                             emb_layer=params['meta']['enc_depth'], # 'agg',
                             label=latent_cluster_key,
                             write_to_disk=True)

In [ ]:
Z = adata.obsm[f'cell_emb_layer_{params['meta']['enc_depth']}']

pca = PCA(n_components=min(Z.shape[0], Z.shape[1]))
pca.fit(Z)

explained = pca.explained_variance_ratio_
print("Cumulative variance first 10 dims:", explained[:10].sum())

std_per_dim = np.std(Z, axis=0)   # shape (768,)
print("min std:", std_per_dim.min())
print("max std:", std_per_dim.max())
print("mean std:", std_per_dim.mean())

eigvals = pca.explained_variance_
eigvals = eigvals[eigvals > 0]  # remove zeros

p = eigvals / eigvals.sum()
effective_rank = np.exp(-(p * np.log(p)).sum())
print("Effective rank:", effective_rank)

In [ ]:
Z = adata.obsm[f'neighborhood_emb_layer_{params['meta']['enc_depth']}']

pca = PCA(n_components=min(Z.shape[0], Z.shape[1]))
pca.fit(Z)

explained = pca.explained_variance_ratio_
print("Cumulative variance first 10 dims:", explained[:10].sum())

std_per_dim = np.std(Z, axis=0)   # shape (768,)
print("min std:", std_per_dim.min())
print("max std:", std_per_dim.max())
print("mean std:", std_per_dim.mean())

eigvals = pca.explained_variance_
eigvals = eigvals[eigvals > 0]  # remove zeros

p = eigvals / eigvals.sum()
effective_rank = np.exp(-(p * np.log(p)).sum())
print("Effective rank:", effective_rank)

### 2.2 xhs1000-39b_1p (Xenium Human Skin Eczema)

#### 2.2.1 Define Dataset Config

In [ ]:
# Set dataset params
dataset_name = 'xhs1000-39b_1p'
obs_cols = ['niche_type']
niche_key = 'niche_type'
batch_key = 'sample'
latent_leiden_resolution_cell = 1.0
latent_leiden_resolution_neighborhood = 0.4
spot_sizes = [20] * 3

In [ ]:
params["data"]["dataset_name"] = dataset_name
params["data"]["raw_data_folder_path"] = (
    params["data"]["raw_data_folder_path"].rsplit("/", 1)[0]
    + "/" + dataset_name)
params["data"]["tokenized_data_folder_path"] = str(tokenized_data_folder_path).rsplit("_", 1)[0] + '.dataset'
params["data"]["precomputed_split"] = f'/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/{params["data"]["tokenized_data_folder_path"].split("/")[-1].replace(".dataset", f"_precomputed_split_{dataset_name}.pkl")}'
params['data']['precomputed_n_nonzero_tokens'] = None
params['data']['test_batch_ids'] = [
    '1000_batch11',
    '1000_batch19',
    '1000_batch32']
params['data']['val_batch_ids'] = None
if 'precomputed_epoch_n_nonzero_tokens' in params['data'].keys():
    del(params['data']['precomputed_epoch_n_nonzero_tokens'])
if 'precomputed_epoch_splits' in params['data'].keys():
    del(params['data']['precomputed_epoch_splits'])

#### 2.2.2 Run Model Inference

In [ ]:
adata = run_inference(dataset_name,
                      params,
                      read_checkpoint,
                      obs_cols=obs_cols,
                      emb_layers=[params['meta']['enc_depth']], # list(range(1, params['meta']['enc_depth'] + 1)),
                      top_k=top_k,
                      write_to_disk=True)

In [ ]:
# Filter cells without niche labels
adata = adata[~adata.obs[niche_key].isna()]

#### 2.2.3 Identify Niches

In [ ]:
adata = compute_umap(adata,
                     emb_keys=['neighborhood'], # ['agg'], ['neighborhood']
                     emb_layer=params['meta']['enc_depth'], # 'agg',
                     write_to_disk=True)

In [ ]:
adata, latent_cluster_key = compute_latent_clusters(
    adata,
    emb_keys=['neighborhood'], # ['agg'], ['neighborhood']
    obs_col=niche_key,
    n_unique_vals=12,
    latent_leiden_resolution_cell=latent_leiden_resolution_cell,
    latent_leiden_resolution_neighborhood=latent_leiden_resolution_neighborhood,
    write_to_disk=True)

#### 2.2.4 Plot Embedding Similarity Stats

In [ ]:
compute_embedding_similarity(adata,
                             emb_keys=['neighborhood'], # ['neighborhood'], ['agg']
                             emb_layer=params['meta']['enc_depth'], # 'agg',
                             label=latent_cluster_key,
                             write_to_disk=True)

In [ ]:
Z = adata.obsm[f'cell_emb_layer_{params['meta']['enc_depth']}']

pca = PCA(n_components=min(Z.shape[0], Z.shape[1]))
pca.fit(Z)

explained = pca.explained_variance_ratio_
print("Cumulative variance first 10 dims:", explained[:10].sum())

std_per_dim = np.std(Z, axis=0)   # shape (768,)
print("min std:", std_per_dim.min())
print("max std:", std_per_dim.max())
print("mean std:", std_per_dim.mean())

eigvals = pca.explained_variance_
eigvals = eigvals[eigvals > 0]  # remove zeros

p = eigvals / eigvals.sum()
effective_rank = np.exp(-(p * np.log(p)).sum())
print("Effective rank:", effective_rank)

In [ ]:
Z = adata.obsm[f'neighborhood_emb_layer_{params['meta']['enc_depth']}']

pca = PCA(n_components=min(Z.shape[0], Z.shape[1]))
pca.fit(Z)

explained = pca.explained_variance_ratio_
print("Cumulative variance first 10 dims:", explained[:10].sum())

std_per_dim = np.std(Z, axis=0)   # shape (768,)
print("min std:", std_per_dim.min())
print("max std:", std_per_dim.max())
print("mean std:", std_per_dim.mean())

eigvals = pca.explained_variance_
eigvals = eigvals[eigvals > 0]  # remove zeros

p = eigvals / eigvals.sum()
effective_rank = np.exp(-(p * np.log(p)).sum())
print("Effective rank:", effective_rank)

### 2.3 xhk1007-CV1-41b_1p (Xenium Human Kidney RCC)

#### 2.3.1 Define Config

In [ ]:
# Set dataset params
dataset_name = 'xhk1007-41b_1p'
cell_type_key = 'level_1_cell_type'
batch_key = 'batch'
latent_leiden_resolution_cell = 0.75
latent_leiden_resolution_neighborhood = 0.2
spot_sizes = [30] * 2

In [ ]:
params["data"]["dataset_name"] = dataset_name
params["data"]["raw_data_folder_path"] = (
    params["data"]["raw_data_folder_path"].rsplit("/", 1)[0]
    + "/xhk1007-CV1-41b_1p")
params["data"]["tokenized_data_folder_path"] = str(tokenized_data_folder_path).rsplit("_", 1)[0] + '.dataset'
params["data"]["precomputed_split"] = f'/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/{params["data"]["tokenized_data_folder_path"].split("/")[-1].replace(".dataset", f"_precomputed_split_{dataset_name}.pkl")}'
params['data']['precomputed_n_nonzero_tokens'] = None
params['data']['test_batch_ids'] = [
     '1007_batch1',
     '1007_batch14'
]
params['data']['val_batch_ids'] = None
if 'precomputed_epoch_n_nonzero_tokens' in params['data'].keys():
    del(params['data']['precomputed_epoch_n_nonzero_tokens'])
if 'precomputed_epoch_splits' in params['data'].keys():
    del(params['data']['precomputed_epoch_splits'])

#### 2.3.2 Run Model Inference

In [ ]:
adata = run_inference(dataset_name,
                      params,
                      read_checkpoint,
                      obs_cols=[cell_type_key],
                      #n_included_cells_list=[1,11], # [i for i in range(1, 12)],
                      emb_layers=[params['meta']['enc_depth']], # list(range(1, params['meta']['enc_depth'] + 1)),
                      top_k=top_k,
                      write_to_disk=True)

#### 2.3.3 Identify Cell Types

In [ ]:
adata = compute_umap(adata,
                     emb_keys=['cell'], # ['agg'], ['neighborhood']
                     emb_layer=params['meta']['enc_depth'], # 'agg',
                     write_to_disk=True)

In [ ]:
adata, latent_cluster_key = compute_latent_clusters(
    adata,
    emb_keys=['cell'], # ['agg'], ['neighborhood']
    obs_col=cell_type_key,
    n_unique_vals=5,
    latent_leiden_resolution_cell=latent_leiden_resolution_cell,
    latent_leiden_resolution_neighborhood=latent_leiden_resolution_neighborhood,
    write_to_disk=True)

#### 2.3.4 Plot Embedding Similarity Stats

In [ ]:
compute_embedding_similarity(adata,
                             emb_keys=['cell'], # ['neighborhood'], ['agg']
                             emb_layer=params['meta']['enc_depth'], # 'agg',
                             label=latent_cluster_key,
                             write_to_disk=True)

In [ ]:
Z = adata.obsm[f'cell_emb_layer_{params['meta']['enc_depth']}']

pca = PCA(n_components=min(Z.shape[0], Z.shape[1]))
pca.fit(Z)

explained = pca.explained_variance_ratio_
print("Cumulative variance first 10 dims:", explained[:10].sum())

std_per_dim = np.std(Z, axis=0)   # shape (768,)
print("min std:", std_per_dim.min())
print("max std:", std_per_dim.max())
print("mean std:", std_per_dim.mean())

eigvals = pca.explained_variance_
eigvals = eigvals[eigvals > 0]  # remove zeros

p = eigvals / eigvals.sum()
effective_rank = np.exp(-(p * np.log(p)).sum())
print("Effective rank:", effective_rank)

In [ ]:
Z = adata.obsm[f'neighborhood_emb_layer_{params['meta']['enc_depth']}']

pca = PCA(n_components=min(Z.shape[0], Z.shape[1]))
pca.fit(Z)

explained = pca.explained_variance_ratio_
print("Cumulative variance first 10 dims:", explained[:10].sum())

std_per_dim = np.std(Z, axis=0)   # shape (768,)
print("min std:", std_per_dim.min())
print("max std:", std_per_dim.max())
print("mean std:", std_per_dim.mean())

eigvals = pca.explained_variance_
eigvals = eigvals[eigvals > 0]  # remove zeros

p = eigvals / eigvals.sum()
effective_rank = np.exp(-(p * np.log(p)).sum())
print("Effective rank:", effective_rank)

### 2.4 xhb1002-75b_3p (Xenium Human Brain GBM)

#### 2.4.1 Define Config

In [ ]:
# Set dataset params
dataset_name = 'xhb1002-75b_3p'
cell_type_key = 'leiden_log1pnorm_res-1'
batch_key = 'batch'
latent_leiden_resolution_cell = 1.0
latent_leiden_resolution_neighborhood = 0.4
spot_sizes = [30] * 3

In [ ]:
params["data"]["dataset_name"] = dataset_name
params["data"]["raw_data_folder_path"] = (
    params["data"]["raw_data_folder_path"].rsplit("/", 1)[0]
    + "/xhb1002-AT9-6b_1p")
params["data"]["tokenized_data_folder_path"] = str(tokenized_data_folder_path).rsplit("_", 1)[0] + '.dataset'
params["data"]["precomputed_split"] = f'/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/{params["data"]["tokenized_data_folder_path"].split("/")[-1].replace(".dataset", f"_precomputed_split_{dataset_name}.pkl")}'
params['data']['precomputed_n_nonzero_tokens'] = None
params['data']['test_batch_ids'] = [
    '1002_batch3',
    '1002_batch15',
    '1002_batch27']
params['data']['val_batch_ids'] = None
if 'precomputed_epoch_n_nonzero_tokens' in params['data'].keys():
    del(params['data']['precomputed_epoch_n_nonzero_tokens'])
if 'precomputed_epoch_splits' in params['data'].keys():
    del(params['data']['precomputed_epoch_splits'])

#### 2.4.2 Run Model Inference

In [ ]:
adata = run_inference(dataset_name,
                      params,
                      read_checkpoint,
                      obs_cols=[cell_type_key],
                      #n_included_cells_list=[1,11], # [i for i in range(1, 12)],
                      emb_layers=[params['meta']['enc_depth']], # list(range(1, params['meta']['enc_depth'] + 1)),
                      top_k=top_k,
                      write_to_disk=True)

#### 2.4.3 Identify Cell Types

In [ ]:
adata = compute_umap(adata,
                     emb_keys=['cell'], # ['agg'], ['neighborhood']
                     emb_layer=params['meta']['enc_depth'], # 'agg',
                     write_to_disk=True)

In [ ]:
adata, latent_cluster_key = compute_latent_clusters(
    adata,
    emb_keys=['cell'], # ['agg'], ['neighborhood']
    obs_col=cell_type_key,
    n_unique_vals=12,
    latent_leiden_resolution_cell=latent_leiden_resolution_cell,
    latent_leiden_resolution_neighborhood=latent_leiden_resolution_neighborhood,
    write_to_disk=True)

#### 2.4.4 Plot Embedding Similarity Stats

In [ ]:
compute_embedding_similarity(adata,
                             emb_keys=['cell'], # ['neighborhood'], ['agg']
                             emb_layer=params['meta']['enc_depth'], # 'agg',
                             label=latent_cluster_key,
                             write_to_disk=True)

In [ ]:
Z = adata.obsm[f'cell_emb_layer_{params['meta']['enc_depth']}']

pca = PCA(n_components=min(Z.shape[0], Z.shape[1]))
pca.fit(Z)

explained = pca.explained_variance_ratio_
print("Cumulative variance first 10 dims:", explained[:10].sum())

std_per_dim = np.std(Z, axis=0)   # shape (768,)
print("min std:", std_per_dim.min())
print("max std:", std_per_dim.max())
print("mean std:", std_per_dim.mean())

eigvals = pca.explained_variance_
eigvals = eigvals[eigvals > 0]  # remove zeros

p = eigvals / eigvals.sum()
effective_rank = np.exp(-(p * np.log(p)).sum())
print("Effective rank:", effective_rank)

In [ ]:
Z = adata.obsm[f'neighborhood_emb_layer_{params['meta']['enc_depth']}']

pca = PCA(n_components=min(Z.shape[0], Z.shape[1]))
pca.fit(Z)

explained = pca.explained_variance_ratio_
print("Cumulative variance first 10 dims:", explained[:10].sum())

std_per_dim = np.std(Z, axis=0)   # shape (768,)
print("min std:", std_per_dim.min())
print("max std:", std_per_dim.max())
print("mean std:", std_per_dim.mean())

eigvals = pca.explained_variance_
eigvals = eigvals[eigvals > 0]  # remove zeros

p = eigvals / eigvals.sum()
effective_rank = np.exp(-(p * np.log(p)).sum())
print("Effective rank:", effective_rank)

### 2.5 chl59-8b_1p (NanoString CosMx Human Lung NSCLC)

#### 2.5.1 Define Config

In [ ]:
# Set dataset params
dataset_name = 'chl59-8b_1p'
niche_key = 'niche'
batch_key = 'batch'
latent_leiden_resolution_cell = 1.0
latent_leiden_resolution_neighborhood = 0.6
spot_sizes = [80] * 6

In [ ]:
params["data"]["dataset_name"] = dataset_name
params["data"]["raw_data_folder_path"] = (
    params["data"]["raw_data_folder_path"].rsplit("/", 1)[0]
    + "/" + dataset_name)
params["data"]["tokenized_data_folder_path"] = str(tokenized_data_folder_path).rsplit("_", 1)[0] + '.dataset'
params["data"]["precomputed_split"] = f'/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/{params["data"]["tokenized_data_folder_path"].split("/")[-1].replace(".dataset", f"_precomputed_split_{dataset_name}.pkl")}'
params['data']['precomputed_n_nonzero_tokens'] = None
params['data']['test_batch_ids'] = [
    '59_batch0',
    #'59_batch3', # unannotated
    '59_batch6',
    '59_batch7',
    #'59_batch2', # unannotated
    '59_batch4',
    '59_batch1',
    '59_batch5']
params['data']['val_batch_ids'] = None
if 'precomputed_epoch_n_nonzero_tokens' in params['data'].keys():
    del(params['data']['precomputed_epoch_n_nonzero_tokens'])
if 'precomputed_epoch_splits' in params['data'].keys():
    del(params['data']['precomputed_epoch_splits'])

#### 2.5.2 Run Model Inference

In [ ]:
adata = run_inference(dataset_name,
                      params,
                      read_checkpoint,
                      obs_cols=[niche_key],
                      #n_included_cells_list=[1,11], # [i for i in range(1, 12)],
                      emb_layers=[params['meta']['enc_depth']], # list(range(1, params['meta']['enc_depth'] + 1)),
                      top_k=top_k,
                      write_to_disk=True)

#### 2.5.3 Identify Niches

In [ ]:
adata = compute_umap(adata,
                     emb_keys=['neighborhood'], # ['agg'], ['neighborhood']
                     emb_layer=params['meta']['enc_depth'], # 'agg',
                     write_to_disk=True)

In [ ]:
adata, latent_cluster_key = compute_latent_clusters(
    adata,
    emb_keys=['neighborhood'], # ['agg'], ['neighborhood']
    obs_col=niche_key,
    n_unique_vals=12,
    latent_leiden_resolution_cell=latent_leiden_resolution_cell,
    latent_leiden_resolution_neighborhood=latent_leiden_resolution_neighborhood,
    write_to_disk=True)

#### 2.5.4 Plot Embedding Similarity Stats

In [ ]:
compute_embedding_similarity(adata,
                             emb_keys=['neighborhood'], # ['neighborhood'], ['agg']
                             emb_layer=params['meta']['enc_depth'], # 'agg',
                             label=latent_cluster_key,
                             write_to_disk=True)

In [ ]:
Z = adata.obsm[f'cell_emb_layer_{params['meta']['enc_depth']}']

pca = PCA(n_components=min(Z.shape[0], Z.shape[1]))
pca.fit(Z)

explained = pca.explained_variance_ratio_
print("Cumulative variance first 10 dims:", explained[:10].sum())

std_per_dim = np.std(Z, axis=0)   # shape (768,)
print("min std:", std_per_dim.min())
print("max std:", std_per_dim.max())
print("mean std:", std_per_dim.mean())

eigvals = pca.explained_variance_
eigvals = eigvals[eigvals > 0]  # remove zeros

p = eigvals / eigvals.sum()
effective_rank = np.exp(-(p * np.log(p)).sum())
print("Effective rank:", effective_rank)

In [ ]:
Z = adata.obsm[f'neighborhood_emb_layer_{params['meta']['enc_depth']}']

pca = PCA(n_components=min(Z.shape[0], Z.shape[1]))
pca.fit(Z)

explained = pca.explained_variance_ratio_
print("Cumulative variance first 10 dims:", explained[:10].sum())

std_per_dim = np.std(Z, axis=0)   # shape (768,)
print("min std:", std_per_dim.min())
print("max std:", std_per_dim.max())
print("mean std:", std_per_dim.mean())

eigvals = pca.explained_variance_
eigvals = eigvals[eigvals > 0]  # remove zeros

p = eigvals / eigvals.sum()
effective_rank = np.exp(-(p * np.log(p)).sum())
print("Effective rank:", effective_rank)

### 2.6 chr78-11b_1p (ISS Cartana Human Reproductive Tract)

#### 2.6.1 Define Config

In [ ]:
# Set dataset params
dataset_name = 'chr78-11b_1p'
niche_key = 'broad_anatomy'
batch_key = 'batch'
latent_leiden_resolution_cell = 1.0
latent_leiden_resolution_neighborhood = 0.4
spot_sizes = [1920 * 0.093] * 11

In [ ]:
params["data"]["dataset_name"] = dataset_name
params["data"]["raw_data_folder_path"] = (
    params["data"]["raw_data_folder_path"].rsplit("/", 1)[0]
    + "/" + dataset_name)
params["data"]["tokenized_data_folder_path"] = str(tokenized_data_folder_path).rsplit("_", 1)[0] + '.dataset'
params["data"]["precomputed_split"] = f'/lustre/scratch126/cellgen/lotfollahi/DATASETS/tokenizer/{params["data"]["tokenized_data_folder_path"].split("/")[-1].replace(".dataset", f"_precomputed_split_{dataset_name}.pkl")}'
params['data']['precomputed_n_nonzero_tokens'] = None
params['data']['test_batch_ids'] = [
    '78_batch3',
    '78_batch10',
    '78_batch1',
    '78_batch0',
    '78_batch2',
    '78_batch6',
    '78_batch7',
    '78_batch9',
    '78_batch8',
    '78_batch4',
    '78_batch5']
params['data']['val_batch_ids'] = None
if 'precomputed_epoch_n_nonzero_tokens' in params['data'].keys():
    del(params['data']['precomputed_epoch_n_nonzero_tokens'])
if 'precomputed_epoch_splits' in params['data'].keys():
    del(params['data']['precomputed_epoch_splits'])

#### 2.6.2 Run Model Inference

In [ ]:
adata = run_inference(dataset_name,
                      params,
                      read_checkpoint,
                      obs_cols=[niche_key],
                      #n_included_cells_list=[1,11], # [i for i in range(1, 12)],
                      emb_layers=[params['meta']['enc_depth']], # list(range(1, params['meta']['enc_depth'] + 1)),
                      top_k=top_k,
                      write_to_disk=True)

#### 2.6.3 Identify Niches

In [ ]:
adata = compute_umap(adata,
                     emb_keys=['neighborhood'], # ['agg'], ['neighborhood']
                     emb_layer=params['meta']['enc_depth'], # 'agg',
                     write_to_disk=True)

In [ ]:
adata, latent_cluster_key = compute_latent_clusters(
    adata,
    emb_keys=['neighborhood'], # ['agg'], ['neighborhood']
    obs_col=niche_key,
    n_unique_vals=21,
    latent_leiden_resolution_cell=latent_leiden_resolution_cell,
    latent_leiden_resolution_neighborhood=latent_leiden_resolution_neighborhood,
    write_to_disk=True)

#### 2.6.4 Plot Embedding Similarity Stats

In [ ]:
compute_embedding_similarity(adata,
                             emb_keys=['neighborhood'], # ['neighborhood'], ['agg']
                             emb_layer=params['meta']['enc_depth'], # 'agg',
                             label=latent_cluster_key,
                             write_to_disk=True)

In [ ]:
Z = adata.obsm[f'cell_emb_layer_{params['meta']['enc_depth']}']

pca = PCA(n_components=min(Z.shape[0], Z.shape[1]))
pca.fit(Z)

explained = pca.explained_variance_ratio_
print("Cumulative variance first 10 dims:", explained[:10].sum())

std_per_dim = np.std(Z, axis=0)   # shape (768,)
print("min std:", std_per_dim.min())
print("max std:", std_per_dim.max())
print("mean std:", std_per_dim.mean())

eigvals = pca.explained_variance_
eigvals = eigvals[eigvals > 0]  # remove zeros

p = eigvals / eigvals.sum()
effective_rank = np.exp(-(p * np.log(p)).sum())
print("Effective rank:", effective_rank)

In [ ]:
Z = adata.obsm[f'neighborhood_emb_layer_{params['meta']['enc_depth']}']

pca = PCA(n_components=min(Z.shape[0], Z.shape[1]))
pca.fit(Z)

explained = pca.explained_variance_ratio_
print("Cumulative variance first 10 dims:", explained[:10].sum())

std_per_dim = np.std(Z, axis=0)   # shape (768,)
print("min std:", std_per_dim.min())
print("max std:", std_per_dim.max())
print("mean std:", std_per_dim.mean())

eigvals = pca.explained_variance_
eigvals = eigvals[eigvals > 0]  # remove zeros

p = eigvals / eigvals.sum()
effective_rank = np.exp(-(p * np.log(p)).sum())
print("Effective rank:", effective_rank)